In [12]:
import duckdb
import pandas
import numpy

con = duckdb.connect()

contracts = con.read_csv("datasets/contracts_all.csv")
materials = con.read_csv("datasets/material_deliveries_all.csv")
laborlogs = con.read_csv("datasets/labor_logs_all.csv")

contracts_df = contracts.df()
contracts_df.head()

lldf = laborlogs.df()

con.execute("""
CREATE TABLE labor AS SELECT * FROM read_csv_auto('datasets/labor_logs_all.csv');
""")

con.execute("""
CREATE TABLE role_mapping AS SELECT * FROM read_csv_auto('role_mapping.csv');
""")

cleaned = con.execute("""
SELECT
    l.*,
    COALESCE(m.standard_role, l.role) AS standardized_role
FROM labor l
LEFT JOIN role_mapping m
ON l.role = m.original_role
""").fetchdf()
print(cleaned.columns)
cleaned["standardized_role"].value_counts()


Index(['project_id', 'log_id', 'date', 'employee_id', 'role', 'sov_line_id',
       'hours_st', 'hours_ot', 'hourly_rate', 'burden_multiplier', 'work_area',
       'cost_code', 'standardized_role'],
      dtype='object')


standardized_role
Apprentice                365152
Journeyman Pipefitter     183562
Journeyman Sheet Metal    183155
Helper/Laborer            118076
Controls Technician       117472
Foreman                   117318
Insulator                 117304
Name: count, dtype: int64